# Build the Schedule Everyone Uses

**Topic:** Learning-rate schedules &nbsp;·&nbsp; **Difficulty:** Easy

Linear warmup into cosine decay is the default schedule for basically every large
model trained since 2019, and it is thirty lines of arithmetic. Warmup keeps the
first few hundred steps from blowing up while the adaptive moments are still
garbage. Cosine decay anneals smoothly to near zero without the cliff of a step
schedule.

Write it, and check the properties that matter: continuity at the handoff, the
peak in the right place, and a floor that is nearly reached.

## The data (created in the next cell)

- `total_steps = 10000`, `warmup_steps = 500`.
- `base_lr = 3e-4`, `min_lr = 3e-5`.

## Your task

With **NumPy only**, write a schedule `lr(t)` for `t = 0 .. total_steps-1`:

1. For `t < warmup_steps`: linear from `0` up to `base_lr`, using
   `base_lr * (t + 1) / warmup_steps`.
2. For `t >= warmup_steps`: cosine decay from `base_lr` to `min_lr` over the
   remaining steps, with progress
   `p = (t - warmup_steps) / (total_steps - warmup_steps)` and
   `lr = min_lr + 0.5 * (base_lr - min_lr) * (1 + cos(pi * p))`.
3. Set `answers` to a dict with `"lrs"` (length-10000 array), `"peak_lr"`,
   `"peak_step"` (int), `"lr_at_half"` (the value at `t = total_steps // 2`),
   and `"final_lr"`.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the curve is monotone up then monotone down and never exceeds `base_lr`.

In [ ]:
import numpy as np

total_steps = 10000
warmup_steps = 500
base_lr = 3e-4
min_lr = 3e-5
print(f"{total_steps} steps, {warmup_steps} warmup, "
      f"peak {base_lr:g} decaying to {min_lr:g}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("lrs", [10000], 1.642785, -0.03632178743760882, 1e-06),
    ("peak_lr", [], 0.0003, -5.023140444434974e-05, 1e-06),
    ("peak_step", [], 499.0, -83.55156939243508, 1e-06),
    ("lr_at_half", [], 0.00017614821163876486, -2.9493906869919044e-05, 1e-06),
    ("final_lr", [], 3.0000007381698515e-05, -5.02314168041192e-06, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "lrs" in answers:
    _l = np.asarray(answers["lrs"], dtype=float)
    _up = bool(np.all(np.diff(_l[:warmup_steps]) > 0))
    _down = bool(np.all(np.diff(_l[warmup_steps:]) < 0))
    _cap = bool(_l.max() <= base_lr * (1 + 1e-12))
    print(f"  {'warmup rises':<24} {'ok' if _up else 'wrong'}")
    print(f"  {'cosine falls':<24} {'ok' if _down else 'wrong'}")
    print(f"  {'never exceeds base_lr':<24} {'ok' if _cap else 'wrong'}")
    _ok = _ok and _up and _down and _cap

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

The handoff is continuous by construction: at `t = warmup_steps - 1` warmup
reaches exactly `base_lr`, and at `t = warmup_steps` the cosine term has `p = 0`
so `cos(0) = 1` and the value is `min_lr + (base_lr - min_lr) = base_lr`. Getting
the `+1` right in the warmup numerator is what stops step 0 having learning rate
exactly zero, which wastes a step and, with some optimizers, produces a
divide-by-zero in the logs.

```python
import numpy as np

t = np.arange(total_steps)
lrs = np.empty(total_steps, dtype=float)

warm = t < warmup_steps
lrs[warm] = base_lr * (t[warm] + 1) / warmup_steps

p = (t[~warm] - warmup_steps) / (total_steps - warmup_steps)
lrs[~warm] = min_lr + 0.5 * (base_lr - min_lr) * (1 + np.cos(np.pi * p))

answers = {
    "lrs": lrs,
    "peak_lr": float(lrs.max()),
    "peak_step": int(np.argmax(lrs)),
    "lr_at_half": float(lrs[total_steps // 2]),
    "final_lr": float(lrs[-1]),
}
```